# 4.3 Export optimizer-ready prices

Run after `4_1_generate_test_predictions.ipynb`. This notebook reads the permanent origin-by-horizon forecast archive, applies the selected operational cadence and overlap rule, and writes the optimizer's required `Date,<region>_price` CSV. Changing these settings does not rerun model or heuristic prediction generation.

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../../"))
from EPF import variables
sys.path.append(os.path.abspath("."))
import forecast_output

In [2]:
OPTIMIZER_EXPORT_METHOD = "value_signal"  # predicted, value_signal, naive, or operating_protocol
FORECAST_CADENCE = "24h"               # must match the optimizer rolling cadence
OVERLAP = "latest_origin"              # latest_origin, earliest_origin, or mean

optimizer_prices = None
optimizer_vintages = None
export_labels = {"predicted": "ml", "value_signal": "value_signal", "naive": "naive", "operating_protocol": "operating_protocol"}
for method in ("predicted", "value_signal", "naive", "operating_protocol"):
    forecast_archive = forecast_output.read_forecast_archive(
        variables.ACTUAL_VS_PREDICTED_TEST_SET,
        column_prefix=method,
        horizon_count=variables.HORIZON_COUNT,
    )
    method_path = variables.OPTIMIZER_ST_PRICE_PATH.with_name(
        f"{variables.OPTIMIZER_ST_PRICE_PATH.stem}_{export_labels[method]}.csv"
    )
    method_prices = forecast_output.write_optimizer_price_csv(
        forecast_archive,
        method_path,
        column_prefix=method,
        region=variables.TARGET_REGION,
        horizon_count=variables.HORIZON_COUNT,
        horizon_minutes=variables.HORIZON_GRANULARITY_IN_MINUTES,
        forecast_cadence=FORECAST_CADENCE,
        overlap=OVERLAP,
    )
    vintage_path = variables.OPTIMIZER_ST_PRICE_PATH.with_name(
        f"{variables.OPTIMIZER_ST_PRICE_PATH.stem}_{export_labels[method]}_vintages.csv"
    )
    method_vintages = forecast_output.optimizer_price_vintages(
        forecast_archive,
        column_prefix=method,
        region=variables.TARGET_REGION,
        horizon_count=variables.HORIZON_COUNT,
        horizon_minutes=variables.HORIZON_GRANULARITY_IN_MINUTES,
        forecast_cadence=FORECAST_CADENCE,
    )
    method_vintages.to_csv(vintage_path, index=False)
    if method == OPTIMIZER_EXPORT_METHOD:
        optimizer_prices = method_prices
        optimizer_vintages = method_vintages
    del forecast_archive, method_prices, method_vintages

# The optimizer routes A2 to this exact canonical filename.
optimizer_prices.to_csv(variables.OPTIMIZER_ST_PRICE_PATH, index=False)
optimizer_vintage_path = variables.OPTIMIZER_ST_PRICE_PATH.with_name(
    f"{variables.OPTIMIZER_ST_PRICE_PATH.stem}_vintages.csv"
)
optimizer_vintages.to_csv(optimizer_vintage_path, index=False)
print(f"Saved all optimizer-ready variants; canonical A2 uses {OPTIMIZER_EXPORT_METHOD!r}: {variables.OPTIMIZER_ST_PRICE_PATH}")
optimizer_prices.head()

Saved all optimizer-ready variants; canonical A2 uses 'predicted': C:\Users\Daniel\OneDrive\Documents\2_Energy_toolkit\1_ST_price_prediction\EPF\5_Model\Data\5_model_results\A2_future_price_ST_predicted.csv


,Date,nsw_price
0,2022-01-01 00:00:00,118.320930
1,2022-01-01 00:30:00,106.409910
2,2022-01-01 01:00:00,94.913124
3,2022-01-01 01:30:00,89.549660
4,2022-01-01 02:00:00,83.005806
